# HW2 Problem 1 · 12살 처음 배우는 학생을 위한 Taxi Neural Network

**완전 실행판**: 아래 코드는 `.py` 원본의 각 실행문을 그대로 이어 붙였으며, Jupyter에서 실행되도록 입력 파일 경로만 `taxi_trips.parquet`로 고정했다. `#` 뒤의 한국어 주석을 읽으며 한 셀씩 실행하면 된다.

필요한 패키지: `numpy`, `pandas`, `torch`, `pyarrow`. 파일은 노트북과 같은 폴더에 있어야 한다(저장소에서는 `data/taxi_trips.parquet`를 이 폴더로 복사한다). 같은 코드는 `homework_2` 폴더에서 `python run_all.py --problem 1`로도 실행된다.

## 준비운동 — 불확실성을 숫자로 느껴보기

두 택시 길의 평균은 모두 20분이다. 하지만 A는 [19,20,21], B는 [10,20,30]분이라면 얼마나 안정적인지는 다르다. 평균은 **기본 예상 소요 시간**, 표준편차는 **얼마나 많이 흔들리는지**다.

- `μ(x)`는 입력 x에서 예상 시간이다.
- `σ(x)`는 예상 시간이 흔들리는 정도로 항상 양수여야 한다.
- `g(x)`가 두 개의 실수를 내놓으면 첫 번째는 μ, 두 번째는 `softplus`로 양수로 바꿔 σ를 만든다.

정규모델의 음의 로그우도:

$$\ell_i=\tfrac12\log(2\pi)+\log\sigma_i+\tfrac{(y_i-\mu_i)^2}{2\sigma_i^2}$$

이 식의 두 번째 항은 넓은 표준편차를 남용하지 못하게 하고, 마지막 항은 좁은 표준편차로 오차를 과신하지 못하게 한다.

In [ ]:
from math import erf, sqrt, log, pi
mean = 40.0  # 평균 여행 시간 40분
for sd in [5.0, 10.0]:  # 같은 평균이라도 불확실성이 다르면
    z = (45.0 - mean) / sd  # 45분이라는 기준을 표준편차 단위로 환산
    probability = 0.5 * (1.0 + erf(z / sqrt(2.0)))  # 표준정규 CDF Φ(z)
    print(f"평균={mean}, 표준편차={sd}: P(Y<45)={probability:.4f}")


## Brier Score는 무엇일까?

실제 43분이면 사건 `Y<45`가 참이어서 `z=1`이고, 48분이면 거짓이어서 `z=0`이다. 예측확률 p에서 Brier 손실은 `(p-z)^2`이다. `p=.8413`일 때 실제 43분이라면 `(.8413-1)^2≈.0252`, 48분이라면 `.8413^2≈.7078`이다. 

정확도는 p≥0.5인지로 예측을 이진화하므로 확률 0.51과 0.99를 비슷하게 다룬다. 데이터의 약 99.55%가 45분 미만이면 정확도만으로는 어려운 여행의 탐지 품질을 알기 어렵다.

## 코드 1. 1. import와 수치계산 패키지 (nn, softplus, numpy, pandas)

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
import sys  # 터미널에서 받은 데이터 파일 이름을 읽는다.
import math  # 정규분포의 상수 pi와 로그를 사용한다.
import numpy as np  # 재현 가능한 데이터 분할을 만든다.
import pandas as pd  # 열 이름이 있는 택시 표를 읽는다.
import torch  # 배열 계산과 자동미분을 담당한다.
from torch import nn  # 신경망 층들을 가져온다.
from torch.nn.functional import softplus  # 실수를 양수인 표준편차로 바꾼다.



## 코드 2. 2. 실험 시드·데이터 읽기·Train/Validation/Test 분할 및 모델 정의

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
torch.manual_seed(7)  # 같은 실행 조건에서 초기 가중치와 배치 순서를 재현한다.
torch.set_num_threads(2)  # CPU 계산에 두 스레드를 사용한다.
df = pd.read_parquet("taxi_trips.parquet")  # 한 행이 한 여행인 표를 연다.
y = torch.tensor(df.pop("trip_duration").to_numpy(), dtype=torch.float32)  # 정답을 분리하여 입력 누출을 막는다.
X = torch.tensor(df.to_numpy(), dtype=torch.float32)  # 남은 일곱 입력을 N행 7열 텐서로 만든다.
idx = np.random.default_rng(42).permutation(len(y))  # 모든 행 번호를 무작위 순서로 섞는다.
tr, va, te = np.split(idx, [int(.70*len(y)), int(.85*len(y))])  # 훈련, 검증, 테스트에 서로 다른 행을 배정한다.
net = nn.Sequential(nn.Linear(7,64), nn.ReLU(), nn.Linear(64,2))  # 여행마다 은닉 특징 64개로 두 수를 계산한다.
opt = torch.optim.Adam(net.parameters(), lr=.002)  # 가중치별 갱신 크기를 조절하는 Adam을 만든다.


## 코드 3. 3. 평균/표준편차 변환과 Normal NLL 함수

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
def predict(x):  # 여러 여행의 조건부 평균과 표준편차를 돌려준다.
    raw = net(x)  # B개 여행에서 B행 2열의 출력을 얻는다.
    return raw[:,0], softplus(raw[:,1]) + .001  # 첫 열은 분 단위 중심, 둘째 열은 양수로 바꾼 폭이다.
def nll(yb, mu, sd):  # 정규분포로 실제 시간들을 얼마나 잘 설명하는지 계산한다.
    return (.5*math.log(2*math.pi)+sd.log()+.5*((yb-mu)/sd)**2).mean()  # 폭 비용과 표준화 잔차 비용을 평균낸다.


## 코드 4. 4. 학습 루프·AutoGrad·Validation으로 epoch 선택

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
best, wait = math.inf, 0  # 처음에는 모든 유한한 검증 점수가 개선이다; wait는 개선 없이 지난 회차 수다.
for epoch in range(300):  # 훈련 자료를 최대 300번 순회한다; 검증 NLL이 더 줄지 않으면 일찍 멈춘다.
    net.train()  # 모델을 학습 모드로 둔다.
    order = torch.as_tensor(tr)[torch.randperm(len(tr))]  # 훈련 행만 다시 섞는다.
    for batch in order.split(2048):  # 마지막 작은 묶음까지 처리한다.
        opt.zero_grad()  # 앞 배치의 기울기를 지운다.
        loss = nll(y[batch], *predict(X[batch]))  # 별표는 평균과 표준편차를 각각 인자로 전달한다.
        loss.backward()  # 손실에서 각 가중치까지 미분한다.
        opt.step()  # 계산한 기울기로 가중치를 한 번 갱신한다.
    net.eval()  # 검증 모드로 전환한다.
    with torch.no_grad():  # 검증에서는 미분을 기록할 필요가 없다.
        score = nll(y[va], *predict(X[va])).item()  # 검증 손실을 파이썬 숫자로 얻는다.
    print("epoch", epoch+1, "validation NLL", round(score, 6))  # 검증 곡선을 로그에 남긴다.
    if score < best:  # 검증 NLL이 줄었는지 비교한다.
        best, chosen_epoch = score, epoch+1  # 가장 좋은 점수와 회차를 기록한다.
        state = {k:v.detach().clone() for k,v in net.state_dict().items()}  # 다음 갱신과 독립적인 가중치 사본을 만든다.
        wait = 0  # 개선되었으므로 대기 횟수를 0으로 되돌린다.
    else:  # 이번 회차에는 검증 NLL이 줄지 않았다.
        wait += 1  # 개선 없는 회차를 하나 센다.
        if wait >= 30:  # 30회 연속 개선이 없으면 더 학습해도 얻을 것이 없다고 본다.
            break  # 조기 종료(early stopping).
print("epochs run", epoch+1)  # 실제로 학습한 회차 수를 출력한다.


## 코드 5. 5. 가장 좋은 가중치 복원

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
net.load_state_dict(state)  # 검증으로 선택한 모델을 복원한다.


## 코드 6. 6. test 확률·Brier·Accuracy 평가

이 블록을 실행하기 전에 앞의 모든 코드 셀을 위에서 아래로 실행했는지 확인하자. 함수, 변수, 모델, 데이터의 의존성을 생략하지 않았다.

In [ ]:
def report(name, mu, sd):  # 같은 테스트 행에서 각 모델을 평가한다.
    z = y[te] < 45  # 45분 미만이면 참, 정확히 45분이면 거짓이다.
    p = torch.special.ndtr(((45-mu)/sd).double())  # 표준화된 경계 왼쪽 면적을 구한다.
    decision = p >= .5  # 사건을 예측할 확률 기준을 적용한다.
    brier = ((p-z.double())**2).mean()  # 확률과 실제 0 또는 1 사이 제곱거리를 평균낸다.
    acc = (decision == z).double().mean()  # 맞힌 여행의 비율을 구한다.
    late_hits = ((~decision) & (~z)).sum()  # 실제 45분 이상 중 모델도 그렇게 판단한 건수다.
    print(name, float(nll(y[te],mu,sd)), brier.item(), acc.item(), late_hits.item())  # 모델별 평가 결과를 출력한다.
with torch.no_grad():  # 최종 테스트는 학습을 끝낸 뒤 수행한다.
    report("constant", y[tr].mean(), y[tr].std(unbiased=False))  # 훈련 평균과 모집단식 표준편차를 쓰는 기준모델이다.
    report("MLP", *predict(X[te]))  # 여행별로 다른 평균과 폭을 사용한다.
print("epoch, validation NLL", chosen_epoch, best)  # 모델 선택의 근거를 출력한다.


## Test 결과 (`python run_all.py --problem 1`로 다시 실행해도 동일)

| Model | Continuous test NLL | Brier ↓ | Accuracy |
|---|---:|---:|---:|
| Constant Normal | 3.476824 | 0.004495 | 99.5505% |
| Heteroscedastic MLP | 3.280044 | 0.003973 | 99.5505% |

Test split에는 45분 이상 택시가 353건 있었지만 MLP는 그중 1건만 양성으로 검출했다. MLP는 Brier가 좋아졌지만 Accuracy는 같았다. 최종 답안은 이 두 지표가 왜 다른 메시지를 주는지 반드시 설명해야 한다.